# Puneri — new voices (free)
Re-voices every line in the game with **AI4Bharat Indic Parler-TTS**: native Marathi, Hindi and Indian-English voices,
each character with their own voice and feeling. Runs on Colab's **free GPU**. Costs nothing.

**Once, before you start**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Make a free account at huggingface.co, open https://huggingface.co/ai4bharat/indic-parler-tts and click **Agree** (it's approved instantly).
3. huggingface.co → Settings → Access Tokens → **New token** (type *Read*). You'll paste it in step 1 below.

**Then** run the cells in order. Step 3 plays a few sample lines so you can hear them first. Step 4 does all ~2,900 lines
(about 1–2 hours) and saves to your Google Drive as it goes, so if Colab disconnects just run it again: it picks up where it stopped.
Step 5 gives you `puneri_voices.zip`. Put it in the game folder and tell Claude.

In [ ]:
# 1. setup (2–3 minutes). Safe to run again.
!pip -q install git+https://github.com/huggingface/parler-tts.git
# parler-tts pulls in an old protobuf that today's transformers can't import: put the new one back
!pip -q install -U protobuf==5.29.5
!apt -qq install -y ffmpeg > /dev/null
# log in to Hugging Face only if this runtime isn't already (or add HF_TOKEN in Colab's Secrets, the key icon, once)
import os, huggingface_hub
try:
    from google.colab import userdata; os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN') or '')
except Exception: pass
if not (os.environ.get('HF_TOKEN') or huggingface_hub.get_token()):
    huggingface_hub.notebook_login()
else: print('Hugging Face: logged in')

In [ ]:
# 2. the model, the lines, and a folder on your Drive to save into
import torch, json, os, subprocess, numpy as np, soundfile as sf
from google.colab import drive
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/puneri_voices'; os.makedirs(OUT, exist_ok=True)
!wget -q -O /content/voice_jobs.json https://raw.githubusercontent.com/once-human/nh48/main/tools/voice_jobs.json
JOBS = json.load(open('/content/voice_jobs.json'))
dev = 'cuda'
model = ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts', torch_dtype=torch.float16).to(dev)
tok = AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts')
dtok = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SR = model.config.sampling_rate
print(len(JOBS), 'lines; sample rate', SR)

# trim silence, a little compression and loudness so lines sit above the street noise, small mono mp3
AF = ('silenceremove=start_periods=1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse,'
      'highpass=f=80,acompressor=threshold=-20dB:ratio=2.5:attack=5:release=80,loudnorm=I=-16:TP=-1.5:LRA=9,apad=pad_dur=0.08')
def save(jid, audio):
    wav = f'/content/{jid}.wav'; sf.write(wav, audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-af', AF, '-ac', '1', '-ar', '24000', '-b:a', '40k', f'{OUT}/{jid}.mp3'], check=True)
    os.remove(wav)

def speak(batch):
    d = dtok([j['desc'] for j in batch], return_tensors='pt', padding=True).to(dev)
    p = tok([j['text'] for j in batch], return_tensors='pt', padding=True).to(dev)
    g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask, prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask,
                       do_sample=True, temperature=0.9, return_dict_in_generate=True)
    out = []
    for i in range(len(batch)):
        n = int(g.audios_length[i]) if getattr(g, 'audios_length', None) is not None else g.sequences.shape[1]
        out.append(g.sequences[i, :n].float().cpu().numpy())
    return out
def sane(j, a):   # a bad take: silence, or far too long for the words
    dur = len(a) / SR
    return 0.25 < dur < 2.0 + len(j['text']) * 0.16 and np.abs(a).max() > 0.02

In [ ]:
# 3. listen first: a few lines from different people
from IPython.display import Audio, display
import random
pick = [j for j in JOBS if j['id'].startswith(('stop_', 's_kaka', 's_kaku', 's_collegegirl', 'k_tapri', 'd_f_mr'))]
random.seed(4); pick = random.sample(pick, 6)
for j, a in zip(pick, speak(pick)):
    print(j['speaker'], '·', j['text']); display(Audio(a, rate=SR))

In [ ]:
# 4. all of them (saved to Drive as it goes; run again after a disconnect and it carries on)
import time
todo = [j for j in JOBS if not os.path.exists(f"{OUT}/{j['id']}.mp3")]
todo.sort(key=lambda j: (j['desc'], len(j['text'])))      # same voice and similar length together: faster batches
print(len(todo), 'to go'); t0 = time.time(); B = 12
for k in range(0, len(todo), B):
    batch = todo[k:k + B]
    try: audios = speak(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); audios = [speak([j])[0] for j in batch]
    for j, a in zip(batch, audios):
        if not sane(j, a): a = speak([j])[0]           # one more take
        if sane(j, a): save(j['id'], a)
    if k // B % 10 == 0: print(f"{k + len(batch)}/{len(todo)}  {(time.time() - t0) / 60:.1f} min")
print('done', len(os.listdir(OUT)), 'files')

In [ ]:
# 5. one zip to download
!cd /content/drive/MyDrive && zip -qr /content/puneri_voices.zip puneri_voices
from google.colab import files
files.download('/content/puneri_voices.zip')